### Importing necessary libraries and API key reference


In [1]:
import os
os.environ.pop("MPLBACKEND", None)   # remove bad value if present
os.environ["MPLBACKEND"] = "Agg"  
import re
import base64
import pandas as pd
import math
import json
import copy
from typing import Any, Dict, List, Optional, Tuple
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from behaviorspace_all_runs import load_behaviorspace_all_runs
from wide_garch_loader import load_wide_timestamp_returns_csv
from conceptual_model_module import build_conceptual_model, score_conceptual_model
from openai import OpenAI

client = OpenAI()


### Reading the CSV file and organising the relevant data
This code refers to a helper module - behaviorspace_all_runs, stored in the same directory as this file

##### Wolf-Sheep Model

In [3]:
"""CSV_FILE = "DataSet/incorrect_wolf_sheep.csv"

all_runs= load_behaviorspace_all_runs(CSV_FILE)
LLM_MAX_STEPS = 100
# Example: use run 0’s time series as the “main” data
run0 = all_runs[0]
example_data = {
    "step": run0["data"]["step"][:LLM_MAX_STEPS],
    "count_sheep": run0["data"]["count_sheep"][:LLM_MAX_STEPS],
    "count_wolves": run0["data"]["count_wolves"][:LLM_MAX_STEPS],
    "count_green_patches": run0["data"]["count_green_patches"][:LLM_MAX_STEPS],
}


experiment_params = run0["params"]"""

'CSV_FILE = "DataSet/incorrect_wolf_sheep.csv"\n\nall_runs= load_behaviorspace_all_runs(CSV_FILE)\nLLM_MAX_STEPS = 100\n# Example: use run 0’s time series as the “main” data\nrun0 = all_runs[0]\nexample_data = {\n    "step": run0["data"]["step"][:LLM_MAX_STEPS],\n    "count_sheep": run0["data"]["count_sheep"][:LLM_MAX_STEPS],\n    "count_wolves": run0["data"]["count_wolves"][:LLM_MAX_STEPS],\n    "count_green_patches": run0["data"]["count_green_patches"][:LLM_MAX_STEPS],\n}\n\n\nexperiment_params = run0["params"]'

##### Flocking Model

In [4]:
"""CSV_FILE = "DataSet/boids_netlogo_jitter_only.csv"
all_runs = load_behaviorspace_all_runs(CSV_FILE)

# Use run 0 as an example
run0 = all_runs[0]

LLM_MAX_STEPS = 100

example_data = {
    "step": run0["data"]["step"][:LLM_MAX_STEPS],
    "alignment": run0["data"]["alignment"][:LLM_MAX_STEPS],
    "cohesion": run0["data"]["cohesion"][:LLM_MAX_STEPS],
    "min-nn-distance": run0["data"]["min-nn-distance"][:LLM_MAX_STEPS],
    "mean-nn-distance": run0["data"]["mean-nn-distance"][:LLM_MAX_STEPS],
    "mean-turn-rate": run0["data"]["mean-turn-rate"][:LLM_MAX_STEPS],
}

experiment_params = run0["params"]"""


'CSV_FILE = "DataSet/boids_netlogo_jitter_only.csv"\nall_runs = load_behaviorspace_all_runs(CSV_FILE)\n\n# Use run 0 as an example\nrun0 = all_runs[0]\n\nLLM_MAX_STEPS = 100\n\nexample_data = {\n    "step": run0["data"]["step"][:LLM_MAX_STEPS],\n    "alignment": run0["data"]["alignment"][:LLM_MAX_STEPS],\n    "cohesion": run0["data"]["cohesion"][:LLM_MAX_STEPS],\n    "min-nn-distance": run0["data"]["min-nn-distance"][:LLM_MAX_STEPS],\n    "mean-nn-distance": run0["data"]["mean-nn-distance"][:LLM_MAX_STEPS],\n    "mean-turn-rate": run0["data"]["mean-turn-rate"][:LLM_MAX_STEPS],\n}\n\nexperiment_params = run0["params"]'

##### Finance Model

In [2]:
CSV_FILE = "DataSet/garch_returns_dataset.csv"

all_runs = load_wide_timestamp_returns_csv(CSV_FILE)

omega = 1e-6
alpha_range = (0.01, 0.09)
beta_range = (0.01, 0.09)
ndata = 1000000

GARCH_PARAMS = {
    "model": "GARCH(1,1)",
    "omega": omega,
    "alpha_range": alpha_range,
    "beta_range": beta_range,
    "ndata": ndata,
    "parameter_schedule": "ramp-up then ramp-down",
    "shock_distribution": "Gaussian",
}

for run in all_runs:
    run["params"] = GARCH_PARAMS

# Use run 0 as an example
run0 = all_runs[0]

LLM_MAX_STEPS = 250

example_data = {
    "t": run0["data"]["step"][:LLM_MAX_STEPS],
    "returns": run0["data"]["returns"][:LLM_MAX_STEPS],
}


experiment_params = run0["params"]


#### Adding the Aggregate run to the original run

In [6]:
def compute_batch_mean_data(
    all_runs: List[Dict[str, Any]],
    run_indices: List[int],
) -> Dict[str, List[float]]:
    """
    Compute pointwise mean of the time-series columns across the runs
    listed in `run_indices`.

    Assumes:
    - All runs have the same set of keys in ['data'].
    - For each key, lists are of equal length across runs.
    """
    if not run_indices:
        raise ValueError("run_indices must be non-empty to compute batch mean.")

    # Use the first run in the batch as template
    first_data = all_runs[run_indices[0]]["data"]
    keys = list(first_data.keys())

    # Length of the time series
    length = len(next(iter(first_data.values())))

    # Initialise accumulators
    mean_data: Dict[str, List[float]] = {k: [0.0] * length for k in keys}

    # Accumulate sums
    for idx in run_indices:
        run_data = all_runs[idx]["data"]
        for k in keys:
            vals = run_data[k]
            if len(vals) != length:
                raise ValueError(f"Run {idx} column '{k}' has inconsistent length.")
            for t in range(length):
                mean_data[k][t] += float(vals[t])

    # Divide by number of runs to get mean
    n = float(len(run_indices))
    for k in keys:
        mean_data[k] = [v / n for v in mean_data[k]]

    return mean_data

# ---------------------------------------------------------
# NEW: Create a deterministic "mean behavior" run
# ---------------------------------------------------------

num_original_runs = len(all_runs)

# Indices of original stochastic runs
original_run_indices = list(range(num_original_runs))

# Compute point-wise mean across all runs
mean_run_data = compute_batch_mean_data(
    all_runs=all_runs,
    run_indices=original_run_indices,
)

# Construct the mean run object
mean_run = {
    "run_index": num_original_runs,  # next index (e.g., 20 if there were 20 runs)
    "data": mean_run_data,
    "params": experiment_params,
    "meta": {
        "type": "batch_mean",
        "description": "Point-wise mean of all stochastic runs"
    }
}

# Append to all_runs
all_runs.append(mean_run)

print(f"Added batch-mean run as run_index={mean_run['run_index']}")

num_original_runs = len(all_runs)


Added batch-mean run as run_index=20


### Simulation Description

In [7]:
simulation_description = f"""
## Simulation description: Time-varying volatility asset return model

This simulation generates synthetic **asset returns** from a stochastic process intended to reproduce key empirical features of financial markets: **volatility clustering** and **time-varying risk**. Returns are expected to show weak serial dependence in levels, while volatility evolves over time, producing alternating periods of calm and turbulence.

Each run outputs a univariate time series of:
- **timestamp / step** (t = 0, …, T−1)
- **returns** (r_t)

The model includes a time-varying volatility mechanism whose persistence and shock sensitivity change deterministically over the run. Specifically, the strength of volatility feedback is designed to **increase during the first half of the run and decrease during the second half**, inducing a controlled rise-and-fall pattern in volatility and tail risk.

### Expected behaviors
A credible analysis should detect:
- weak serial correlation in returns but dependence in squared or absolute returns,
- increasing then decreasing volatility over time,
- corresponding widening and narrowing of tail risk (e.g., rolling kurtosis or extreme quantiles),
- consistency of these qualitative patterns across multiple stochastic runs,
- The simulation model is also supposed to exhibit the leverage effect, according to which negative returns lead to higher future volatility than positive returns.

### Framework goal
The credibility framework should (i) identify volatility clustering and the designed regime change from raw returns, (ii) assess robustness across runs, and (iii) assess whether the leverage effect is present in the generated dynamics.

### What will be judged for credibility assessment

Credibility for this simulation is assessed by evaluating whether the generated return time series exhibits the **intended statistical behaviors**, rather than convergence to a terminal state. Specifically, the framework will judge:

1. **Volatility clustering**  
   Whether returns show weak serial correlation in levels but clear dependence in squared or absolute returns, indicating clustering of volatility over time.

2. **Designed regime change**  
   Whether volatility and tail risk systematically increase during the first part of the run and decrease later, consistent with the stated time-varying schedule.

3. **Time-varying tail behavior**  
   Whether rolling kurtosis or extreme-quantile measures widen during high-volatility periods and narrow during low-volatility periods.

4. **Robustness across runs**  
   Whether the above qualitative patterns are consistently observed across multiple stochastic runs, despite differences in individual return paths.

5. **Leverage effect**  
   Whether negative returns are followed by a systematically stronger increase in volatility than positive returns of similar magnitude.

Together, these criteria evaluate whether the simulation credibly reproduces the statistical properties it claims to model.
"""


### Conceptual model Layer

In [8]:
# === Conceptual modeling layer (model-level) ===

data_columns = list(example_data.keys())
param_names = list(experiment_params.keys())

# 1) Build conceptual model
conceptual_model = build_conceptual_model(
    simulation_description=simulation_description,
    data_columns=data_columns,
    param_names=param_names,
)

print("\n=== Conceptual Model (inferred) ===\n")
print(json.dumps(conceptual_model, indent=2))

"""# 2) Score conceptual model validity
conceptual_eval = score_conceptual_model(conceptual_model)
conceptual_validity_score = float(conceptual_eval.get("validity_score", 0.0))
conceptual_validity_reasoning = conceptual_eval.get("reasoning", "")

print("\n=== Conceptual Model Evaluation ===")
print(f"Conceptual validity score: {conceptual_validity_score:.3f}")
print(conceptual_validity_reasoning, "\n")"""


# Convenience: list of expected behaviours for prompts
CONCEPTUAL_EXPECTATIONS_TEXT = "\n".join(
    f"- {b}" for b in conceptual_model.get("expected_behaviours", [])
)



=== Conceptual Model (inferred) ===

{
  "objectives": [
    "Study the dynamic behavior of returns generated by a parametric model under varying parameter ranges, schedules, and shock distributions.",
    "Assess how different combinations of omega, alpha, and beta parameters influence the temporal evolution and distributional properties of returns.",
    "Evaluate sensitivity of simulated return paths to alternative shock distributions and parameter schedules over a fixed data horizon (ndata)."
  ],
  "entities": [
    {
      "name": "return_process",
      "role": "agent",
      "state_variables": [
        "current_return",
        "latent_volatility",
        "time_index"
      ],
      "columns": [
        "returns"
      ]
    },
    {
      "name": "time",
      "role": "environment",
      "state_variables": [
        "current_time_step"
      ],
      "columns": [
        "t"
      ]
    },
    {
      "name": "shock",
      "role": "resource",
      "state_variables": [
  

## Generates code for plotting the graphs
The code generated is stored as a helper module in the same directory

In [9]:
plotting_prompt = f"""
You are an expert data analyst.

You are given simulation time-series data from a single run.
The data is provided as a Python dict mapping column names to lists of values, e.g.:

data = {example_data}

1) SIMULATION DESCRIPTION (context):
{simulation_description}

2) CONCEPTUAL MODEL for this simulation (objectives, entities, state variables, mechanisms, assumptions, expected behaviours):
{json.dumps(conceptual_model, indent=2)}

Your task is to write a SMALL HELPER MODULE that defines a single function:

    def generate_plots(data, out_dir: str) -> None:

CRITICAL GOAL:
Create plots that are *most useful for understanding whether the run behaves plausibly*,
based on the simulation description and conceptual model. Do NOT plot everything.

Requirements for generate_plots:

0) Output:
   - Produce EXACTLY 3 figures.
   - Save them as:
       - "graph1.png"
       - "graph2.png"
       - "graph3.png"

1) Inputs and inference:
   - `data` is a dict mapping column names to lists of numeric values.
   - Infer the time axis from keys like: 'step', 't', 'time', 'tick', 'ticks', 'iteration'.
     If multiple exist, choose the most plausible time axis (monotonic increasing is preferred).
   - From the remaining keys, select ONLY the most informative variables for the two plots.
     Do not include irrelevant or redundant variables.

2) Variable selection rules (generic, but strict):
   - Exclude constant or nearly-constant series (very low variance) unless it is clearly meaningful.
   - Prefer variables that:
       (a) represent core system state (counts, totals, averages, distances, energies, errors),
       (b) show clear dynamics (trends, convergence, oscillation, spikes),
       (c) align with the conceptual model’s objectives and expected behaviors.
   - Avoid plotting more than 4 unique variables total across both figures (not counting time).
   - Do not choose variables solely because they exist; justify selection implicitly by the plot design.

3) Plot design rules (strict):
   - (Very Important) The plots must be visually and analytically distinct, using varied plot types and layouts to maximize information density.
   - Do NOT put more than 3 lines on the same axes.
   - If you need more than 2 variables in a figure, use multiple subplots within that figure
     (stacked vertically), sharing the same x-axis.
   - Use clear titles and axis labels that explain what each plot is showing.
   - Use ONLY matplotlib (no seaborn).
   - Do NOT call plt.show().
   - Close each figure with plt.close() after saving.
   - Create `out_dir` if it does not exist.
   



5) Robustness:
   - Handle missing or irregular data gracefully.
   - Convert lists to numpy arrays internally.
   - If lengths do not match the time axis, truncate all series to the shortest length.
   - Ensure the function always produces exactly 2 saved figures.

   Outlier and scale handling (important):
   - If a variable contains extreme early values or isolated spikes that dominate the y-axis
     and visually obscure later dynamics, mitigate this in a transparent way.
   - Prefer one of the following strategies (choose the simplest that preserves interpretability):
       (a) Use a logarithmic y-scale if the values are strictly positive.
       (b) Trim only the initial transient when it is clearly an initialization artifact,
           and document this in the plot title (e.g., "after initial transient").
       (c) Plot the same variable in two vertically stacked subplots:
           – top: full range (including outliers),
           – bottom: zoomed-in range excluding extreme values (e.g., clipping to percentile bounds).
   - Do NOT silently drop data or hide outliers without explanation.
   - Do NOT rescale data in a way that changes its qualitative meaning.


6) You may assume the following imports already exist at module level:
   - import os
   - import numpy as np
   - import matplotlib
   - matplotlib.use("Agg", force=True)
   - import matplotlib.pyplot as plt


7) Do NOT execute anything at import time:
   - matplotlib version is 3.9.2, code must be according to that
   - No top-level plotting.
   - No `if __name__ == '__main__':`
   - Only define the function `generate_plots`.

8) PERFORMANCE CONSTRAINTS (Very IMPORTANT):
- The time series can be very long (up to 1,000,000 steps). 
- Give the code accordingly.
- if any downsampling is required then do it before all the process. The plotting should not get stuck because for longer computing times


Return ONLY the Python code for this helper module, inside a ```python code block,
with the imports and the function definition.
"""


In [10]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

n_tokens = len(enc.encode(plotting_prompt))
print("Tokens:", n_tokens)


Tokens: 5964


In [11]:
response = client.chat.completions.create(
    model="gpt-5.1",
    messages=[{"role": "user", "content": plotting_prompt}],
)

In [12]:
plot_code_raw = response.choices[0].message.content

match = re.search(r"```python(.*?)```", plot_code_raw, re.DOTALL)
plot_code = match.group(1) if match else plot_code_raw

In [13]:
with open("generated_plot_helper.py", "w", encoding="utf-8") as f:
    f.write(plot_code)

print("\nStep 2 complete - helper module generated\n")



Step 2 complete - helper module generated



### Plots graphs for all the runs

In [3]:
import importlib
import generated_plot_helper  # first import
importlib.reload(generated_plot_helper)  # ensure we see latest code

from generated_plot_helper import generate_plots
run_dirs = []

for run in all_runs:
    run_idx = run["run_index"]    # 0-based
    run_data = run["data"]

    out_dir = f"run_{run_idx + 1}"
    run_dirs.append(out_dir)

    generate_plots(run_data, out_dir=out_dir)

print("\nStep 3 complete - graphs generated for all runs via helper module\n")


KeyboardInterrupt: 

## Generates 4 indicators and their weights 

In [19]:
# ask GPT for 4 indicators and their weights, grounded in the conceptual model

indicator_prompt = f"""
You are helping define evaluation criteria for a simulation credibility assessment for a specific simulation model.

You are given:

1) SIMULATION DESCRIPTION:
{simulation_description}

2) CONCEPTUAL MODEL for this simulation (objectives, entities, state variables, mechanisms, assumptions, expected behaviours):
{json.dumps(conceptual_model, indent=2)}



Your task is to propose EXACTLY 4 indicators that will be used to score
INDIVIDUAL RUNS of THIS simulation model.

Requirements for the indicators:
- EACH indicator must correspond to an important aspect of credibility GIVEN the conceptual model above.
- At least some indicators should clearly relate to the expected behaviours and/or the numeric rules (for example, penalizing runs that violate basic assumptions such as non-negativity or simple predator–prey logic).
- Indicators must be reusable across different runs of the same model (do NOT refer to specific run IDs or one-off anomalies).
- Indicators should collectively cover different, non-overlapping aspects of credibility (do not define four nearly-identical indicators).

For EACH indicator, provide:
- "name": a short string name
- "description": one sentence describing what it measures and how it relates to the conceptual model (expectations, mechanisms, or rules)
- "weight": a float between 0.0 and 1.0 indicating importance

IMPORTANT about weights:
- Use the conceptual model to decide which aspects of credibility are MOST critical for this model.
- If violating a basic assumption or rule would seriously undermine the model, the corresponding indicator(s) should have a relatively HIGHER weight.
- Less critical aspects (nice-to-have, secondary behaviours) should have LOWER weights.
- All weights MUST sum to 1.0 (within a small numerical tolerance).
- Do NOT make all weights equal unless you genuinely judge all aspects to be equally important for THIS specific model.

Output JSON ONLY, with this structure:

{{
  "indicators": [
    {{
      "name": "...",
      "description": "...",
      "weight": 0.0
    }},
    {{
      "name": "...",
      "description": "...",
      "weight": 0.0
    }},
    {{
      "name": "...",
      "description": "...",
      "weight": 0.0
    }},
    {{
      "name": "...",
      "description": "...",
      "weight": 0.0
    }}
  ]
}}

Do not add any extra keys or text outside the JSON.
"""

indicator_messages = [
    {
        "role": "system",
        "content": "You are an expert modeler defining transparent, reusable indicators for simulation credibility."
    },
    {
        "role": "user",
        "content": "SIMULATION DESCRIPTION:\n" + simulation_description
    },
    
    {
        "role": "user",
        "content": indicator_prompt
    }
]




indicators_response = client.chat.completions.create(
    model="gpt-5.1",
    messages=indicator_messages,
    max_completion_tokens=600,
)

indicators_raw = indicators_response.choices[0].message.content
indicators_json = json.loads(indicators_raw)
indicators = indicators_json["indicators"]

print("\n indicators and weights \n")
for ind in indicators:
    print(f"- {ind['name']} (weight={ind['weight']:.3f})")
    print(f"  {ind['description']}\n")


 indicators and weights 

- VolatilityClusteringStructure (weight=0.300)
  Assesses whether the run exhibits weak autocorrelation in raw returns but significant positive autocorrelation in squared or absolute returns, as expected from the conditional_variance mechanism and volatility clustering behavior.

- DesignedVolatilityRegimePattern (weight=0.300)
  Evaluates whether rolling volatility (e.g., rolling standard deviation of returns) shows a clear rise during the first half of the run and a decline during the second half, consistent with the deterministic parameter_schedule driving conditional_variance.

- TimeVaryingTailRisk (weight=0.200)
  Checks that rolling tail-risk measures (such as rolling kurtosis or extreme quantiles of returns) widen in periods of elevated volatility and narrow in calmer periods, aligning tail behavior with the model’s time-varying risk mechanism.

- LeverageEffectPresence (weight=0.200)
  Measures whether negative returns are followed by systematically 

## Generates Insights

In [20]:
# helper to encode image as base64 data URL
def encode_image(path: str) -> str:
    with open(path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("utf-8")
    return f"data:image/png;base64,{b64}"



In [ ]:
# list to store insights for all runs
# each item: {"run_index": int, "run_dir": str, "insights": str}
insights_per_run = []

for run_idx, run_dir in enumerate(run_dirs):  # run_dirs already built earlier
    # collect this run's graphs
    image_paths = []
    for gname in ["graph1.png", "graph2.png", "graph3.png"]:
        p = os.path.join(run_dir, gname)
        if os.path.exists(p):
            image_paths.append(p)

    if not image_paths:
        print(f"No graphs found for {run_dir}, skipping.")
        continue

    # NEW: get numeric rule summary for this run
    run_data = all_runs[run_idx]["data"]
    

    # build messages for this run
    messages = []

    # high-level instruction + context, now including expectations + rule summary
    messages.append({
        "role": "user",
        "content": (
            f"These are plots from simulation run {run_idx + 1} "
            f"({run_dir}). Use the simulation description and conceptual expectations as context.\n\n"
            "CONCEPTUAL EXPECTATIONS (behaviour the model claims to have):\n"
            f"{CONCEPTUAL_EXPECTATIONS_TEXT}\n\n"
            "TASK: Provide 4 structured, technical insights about this single run.\n"
            "- First, describe the main patterns you see (stability, oscillations, trends, anomalies).\n"
            "- Second, explicitly discuss how this run SUPPORTS or VIOLATES the conceptual expectations,\n"            
            "Return the insights as a numbered list of short paragraphs."
        ),
    })

    messages.append({
        "role": "user",
        "content": "SIMULATION DESCRIPTION:\n" + simulation_description
    })

    # add each image
    for path in image_paths:
        messages.append({
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": (
                        f"Run {run_idx + 1}, plot file: {os.path.basename(path)}. "
                        "Analyze this in the context of this run and the conceptual expectations."
                    ),
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": encode_image(path),
                    },
                },
            ],
        })

    analysis_response = client.chat.completions.create(
        model="gpt-5.1",
        messages=messages,
        max_completion_tokens=1200,
    )

    insights_text = analysis_response.choices[0].message.content

    insights_per_run.append({
        "run_index": run_idx,
        "run_dir": run_dir,
        "insights": insights_text,   # NEW: keep rule summary with the run
    })

    print(f"\n=== Insights for run {run_idx + 1} ({run_dir}) ===\n")

print("\nStep 4 complete - collected insights for all runs\n")



=== Insights for run 1 (run_1) ===


=== Insights for run 2 (run_2) ===


=== Insights for run 3 (run_3) ===


=== Insights for run 4 (run_4) ===


=== Insights for run 5 (run_5) ===


=== Insights for run 6 (run_6) ===


=== Insights for run 7 (run_7) ===


=== Insights for run 8 (run_8) ===


=== Insights for run 9 (run_9) ===


=== Insights for run 10 (run_10) ===


=== Insights for run 11 (run_11) ===


=== Insights for run 12 (run_12) ===


=== Insights for run 13 (run_13) ===


=== Insights for run 14 (run_14) ===


=== Insights for run 15 (run_15) ===


=== Insights for run 16 (run_16) ===


=== Insights for run 17 (run_17) ===


=== Insights for run 18 (run_18) ===


=== Insights for run 19 (run_19) ===


=== Insights for run 20 (run_20) ===


=== Insights for run 21 (run_21) ===


Step 4 complete - collected insights for all runs



## Feedback Looping

### Helper functions

In [22]:


# NEW: helper to compute a batch-mean "cumulative run" time series

def safe_json_loads(text: str) -> Dict[str, Any]:
    """
    Strict JSON parser with a helpful error message.
    """
    try:
        return json.loads(text)
    except json.JSONDecodeError as e:
        raise ValueError(f"LLM did not return valid JSON. Error: {e}\nRaw:\n{text}")


def compute_batch_credibility_from_run_indicators(
    run_indicator_scores: List[Dict[str, Any]],
    indicators: List[Dict[str, Any]],
    robustness_k: float = 1.0,  # kept in signature for compatibility, now unused
    eps: float = 1e-6,          # kept in signature for compatibility, now unused
) -> Dict[str, Any]:
    """
    SIMPLE batch credibility.

    For each indicator i:
      mean_i = mean score across runs in THIS BATCH (including the batch-mean run).

    Overall batch credibility = weighted arithmetic mean of mean_i:
      overall = sum_i (w_i * mean_i) / sum_i (w_i over indicators that have any scores)

    This uses ONLY:
      - the indicator scores for runs in this batch (run_indicator_scores),
      - and the indicator weights (indicators[*]["weight"]).

    No variance penalty, no geometric mean.
    """
    # Collect scores per indicator name
    name_to_vals: Dict[str, List[float]] = {ind["name"]: [] for ind in indicators}
    name_to_w: Dict[str, float] = {ind["name"]: float(ind["weight"]) for ind in indicators}

    for run in run_indicator_scores:
        for s in run["scores"]:
            nm = s["name"]
            if nm in name_to_vals:
                name_to_vals[nm].append(float(s["score"]))

    per_indicator_batch: List[Dict[str, Any]] = []

    weighted_sum = 0.0
    weight_total = 0.0

    for nm, vals in name_to_vals.items():
        w = name_to_w.get(nm, 0.0)

        if not vals:
            mean_i = 0.0
        else:
            mean_i = sum(vals) / len(vals)

        per_indicator_batch.append({
            "name": nm,
            "weight": w,
            "mean_across_runs": mean_i,
        })

        if w > 0 and vals:
            weighted_sum += w * mean_i
            weight_total += w

    if weight_total <= 0:
        overall = 0.0
    else:
        overall = weighted_sum / weight_total

    return {
        "batch_credibility": float(overall),
        "per_indicator_batch": per_indicator_batch,
        "aggregator": {
            "type": "weighted_mean_of_indicator_means",
            "note": "Batch credibility = weighted arithmetic mean of per-indicator mean scores across runs in this batch (including batch-mean run).",
        },
    }



def normalize_indicator_weights(indicators: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    Ensure weights sum to 1.0 (within float tolerance). If already sum ~1, returns as-is.
    """
    inds = copy.deepcopy(indicators)
    total = sum(float(i["weight"]) for i in inds)
    if total <= 0:
        return inds
    if abs(total - 1.0) < 1e-9:
        return inds
    for i in inds:
        i["weight"] = float(i["weight"]) / total
    return inds


def build_batch_summary_object(
    batch_id: int,
    run_indicator_scores: List[Dict[str, Any]],
    batch_agg: Dict[str, Any],
) -> Dict[str, Any]:
    """
    Compact, loggable object fed into face validation + stored in logs.
    """
    return {
        "batch_id": batch_id,
        "run_indices": [r["run_index"] for r in run_indicator_scores],
        "per_run_indicator_scores": run_indicator_scores,  # already includes reasoning
        "batch_indicator_summary": batch_agg["per_indicator_batch"],
        "batch_credibility": batch_agg["batch_credibility"],
        "aggregator": batch_agg["aggregator"],
    }


def apply_indicator_score_adjustments(
    run_indicator_scores: List[Dict[str, Any]],
    adjustments: List[Dict[str, Any]],
) -> List[Dict[str, Any]]:
    """
    Applies score adjustments returned by the LLM to the per-run indicator scores.

    Expected adjustment item format:
    {
      "run_index": int,
      "indicator_name": str,
      "new_score": float,
      "reason": str,  # should cite issue_id + issue text
    }
    """
    updated = copy.deepcopy(run_indicator_scores)
    index_map = {r["run_index"]: r for r in updated}

    for adj in adjustments:
        ri = adj["run_index"]
        name = adj["indicator_name"]
        new_score = float(adj["new_score"])
        reason = adj.get("reason", "")

        if ri not in index_map:
            continue
        for s in index_map[ri]["scores"]:
            if s["name"] == name:
                s["score"] = max(0.0, min(1.0, new_score))
                # append traceable reason
                s["reasoning"] = (s.get("reasoning", "").strip() + " | ADJUSTED: " + reason).strip()
                break

    return updated


### Prompts

1. Scoring indicators
2. Meta Validation (Critic)
3. Feedback implementation

In [23]:
def prompt_per_run_scoring() -> str:
    return """
You are scoring simulation credibility INDICATORS for a SINGLE RUN.

You will receive:
- SIMULATION DESCRIPTION
- INDICATORS (fixed names + weights)
- CONCEPTUAL EXPECTATIONS (from the model-level conceptual understanding)

- RUN INSIGHTS (which already relate observations to expectations and rule results)

IMPORTANT:
- Output indicator scores FOR THIS RUN ONLY.
- Do NOT compute any batch-level credibility here.

SCORING GUIDANCE:
- Ground every score strictly in the RUN INSIGHTS.


For each indicator, output:
- name: exact indicator name (must match provided list)
- weight: copy the provided weight exactly
- score: 0.0 to 1.0 for THIS RUN
- reasoning: 1–3 sentences grounded in this run’s evidence, explicitly referencing:
    * which conceptual expectations are supported/violated
    * patterns described in RUN INSIGHTS

Output JSON ONLY:
{
  "scores": [
    {"name":"...", "weight":0.0, "score":0.0, "reasoning":"..."}
  ]
}

Rules:
- Use exactly the indicators provided.
- No extra keys, no extra text.
"""


def prompt_face_validation() -> str:
    return """
You are reviewing a batch of simulation credibility assessments as an experienced human modeler.

IMPORTANT: Your job is to CHECK CALIBRATION, not to invent improvements.
- Prefer rubric stability. Do NOT propose changes unless there is clear evidence of miscalibration in THIS BATCH.
- Do NOT "raise the bar" simply because you can.
- Only recommend changes when you can point to specific runs/indicators where the current rubric produces an obviously incorrect or inconsistent result relative to the SIMULATION DESCRIPTION and the provided evidence.

You are given, for THIS BATCH ONLY:
- SIMULATION DESCRIPTION
- CURRENT INDICATORS (names, descriptions, weights)
- Per-run indicator scores (0–1) with reasoning
- One batch-level credibility score (computed deterministically from the set of runs)
- Aggregator description and across-run indicator summary

You MUST output the following JSON ONLY:

{
  "short_summary": "Exactly 2 sentences summarising whether a change is warranted and the biggest issue (if any).",
  "issues_found": [
    {
      "issue_id": "ISSUE-1",
      "issue": "...",
      "evidence": "... (must cite across-run or specific run evidence)",
      "severity": "low|medium|high"
    }
  ],
  "recommend_change": true/false,
  "recommended_changes": [
    {
      "type": "indicator score adjust|weight_adjust|indicator_change",
      "issue_id": "ISSUE-1",
      "details": "Describe the minimal change needed and what it fixes."
    }
  ]
}

Hard rules:
- If no problems: set recommend_change=false, issues_found=[], recommended_changes=[]
- Every recommended_changes item MUST reference an existing issue_id from issues_found.
- No extra keys, no extra text.
"""


def prompt_indicator_score_adjust() -> str:
    return """
You are adjusting PER-RUN INDICATOR SCORES for THIS BATCH ONLY based on a specific face-validation issue.

You are given:
- SIMULATION DESCRIPTION
- CURRENT INDICATORS
- The batch's per-run indicator scores (with reasoning)
- The batch aggregation summary (mean/std per indicator)
- A single TARGET ISSUE to correct (issue_id + issue text + evidence)

Goal:
- Propose the MINIMAL set of score adjustments needed so that the per-run indicator scores better match the rubric and evidence.
- Do NOT change indicator definitions or weights here.
- Do NOT invent new indicators.

Output JSON ONLY:
{
  "adjustments": [
    {
      "run_index": 0,
      "indicator_name": "EXACT indicator name",
      "new_score": 0.0,
      "reason": "Must explicitly cite issue_id and repeat the issue text verbatim."
    }
  ]
}

Rules:
- If no adjustments are truly needed, output {"adjustments": []}
- Keep changes minimal (adjust only the runs/indicators necessary).
- No extra keys, no extra text.
"""


def prompt_indicator_update() -> str:
    return """
You are updating the INDICATOR SET and/or WEIGHTS for scoring FUTURE batches.

CRITICAL RULE: DEFAULT IS NO CHANGE.
- Rubric drift destroys comparability across batches. Keep indicators and weights IDENTICAL unless there is strong, specific evidence of miscalibration.
- Do NOT get stricter over time by default. Minimal change only.

You are given:
- SIMULATION DESCRIPTION
- CURRENT indicators + weights
- Face-validation output with issues_found and recommended_changes

Allowed changes (only if required):
1) indicator_change: clarify wording, split/merge/add/remove indicators ONLY if the current set cannot represent a required credibility concept.
2) weight_adjust: adjust weights ONLY if there is clear evidence that weights mis-rank credibility under the current aggregator.

Constraints for weight_adjust:
- Change at most TWO weights.
- Each weight changes by at most ±0.05.
- All weights in [0,1] and must sum to 1.0 exactly.

Output JSON ONLY:
{
  "indicators": [
    {
      "name": "EXACT indicator name (existing or new)",
      "description": "Updated or unchanged description",
      "weight": 0.0,
      "change_reason": "If unchanged: 'no change (stability)'. If changed: MUST cite the issue by writing: ISSUE_REF: <issue_id> | <issue text verbatim>."
    }
  ]
}

Hard rules:
- If no change warranted, output the FULL list identical to CURRENT, each change_reason = "no change (stability)".
- If you changed anything, every changed indicator MUST cite an issue using ISSUE_REF exactly as specified.
- No extra keys, no extra text.
"""


### Feedback Mechanism Implementation

In [24]:
BATCH_SIZE = 5
ROBUSTNESS_K = 1.0  # higher => stronger penalty for across-run disagreement

# If you start from a preregistered rubric:
current_indicators = normalize_indicator_weights(indicators)

all_batch_results: List[Dict[str, Any]] = []
batch_feedback_log: List[Dict[str, Any]] = []

# CHANGED: use original runs only for batching, cumulative runs will be added per-batch
num_original_runs = len(all_runs)

# NEW: quick lookup from run_index to its insights entry
insights_by_run_index: Dict[int, Dict[str, Any]] = {
    r["run_index"]: r for r in insights_per_run
}

In [25]:
# ----------------------------
# NEW EXECUTION: score ALL runs -> face validate -> if changed, re-score ALL runs -> repeat until stable
# ----------------------------

all_cycle_results: List[Dict[str, Any]] = []
cycle_feedback_log: List[Dict[str, Any]] = []

num_original_runs = len(all_runs)

# quick lookup from run_index to its insights entry (already built earlier)
insights_by_run_index: Dict[int, Dict[str, Any]] = {r["run_index"]: r for r in insights_per_run}

cycle_id = 0

while True:
    cycle_id += 1
    print(f"\n==============================")
    print(f"=== Cycle {cycle_id}: scoring all runs 1–{num_original_runs} ===")
    print(f"==============================\n")

    # 1) LLM scores indicators PER RUN (all original runs)
    run_indicator_scores: List[Dict[str, Any]] = []

    for run_index in range(num_original_runs):
        if run_index not in insights_by_run_index:
            raise ValueError(f"Missing insights for run_index={run_index}. Did Step 4 run?")

        insights_text = insights_by_run_index[run_index]["insights"]

        messages = [
            {"role": "system", "content": "You are an expert simulation evaluator assigning calibrated indicator scores."},
            {"role": "user", "content": "SIMULATION DESCRIPTION:\n" + simulation_description},
            {"role": "user", "content": "INDICATORS (with weights):\n" + json.dumps(current_indicators, indent=2)},
            {"role": "user", "content": "CONCEPTUAL EXPECTATIONS:\n" + CONCEPTUAL_EXPECTATIONS_TEXT},
            {"role": "user", "content": f"RUN INDEX: {run_index}\nRUN INSIGHTS:\n{insights_text}"},
            {"role": "user", "content": prompt_per_run_scoring()},
        ]

        resp = client.chat.completions.create(model="gpt-5.1", messages=messages)
        per_run = safe_json_loads(resp.choices[0].message.content)

        run_indicator_scores.append({
            "run_index": run_index,
            "scores": per_run["scores"],
        })

        print(f"Run {run_index + 1}/{num_original_runs}: indicator scoring done.")

    # 2) Python computes ONE overall credibility across ALL runs
    overall_agg = compute_batch_credibility_from_run_indicators(
        run_indicator_scores=run_indicator_scores,
        indicators=current_indicators,
        robustness_k=ROBUSTNESS_K,
    )

    overall_summary = build_batch_summary_object(
        batch_id=cycle_id,  # reusing the same structure; now "batch_id" = "cycle_id"
        run_indicator_scores=run_indicator_scores,
        batch_agg=overall_agg,
    )

    print(f"\nCycle {cycle_id} overall credibility: {overall_summary['batch_credibility']:.3f}\n")

    # 3) Face validation ONCE over the whole set of runs
    fv_messages = [
        {"role": "system", "content": "You are performing an expert human-style face validity check."},
        {"role": "user", "content": "SIMULATION DESCRIPTION:\n" + simulation_description},
        {"role": "user", "content": "CURRENT INDICATORS:\n" + json.dumps(current_indicators, indent=2)},
        {"role": "user", "content": "BATCH SUMMARY:\n" + json.dumps(overall_summary, indent=2)},
        {"role": "user", "content": prompt_face_validation()},
    ]

    fv_resp = client.chat.completions.create(model="gpt-5.1", messages=fv_messages)
    fv_json = safe_json_loads(fv_resp.choices[0].message.content)

    print(f"Face validation (cycle {cycle_id}) summary:\n{fv_json['short_summary']}\n")

    # log this cycle’s results (audit)
    all_cycle_results.append(copy.deepcopy(overall_summary))
    cycle_feedback_log.append({
        "cycle_id": cycle_id,
        "run_indices": overall_summary["run_indices"],
        "overall_credibility": overall_summary["batch_credibility"],
        "aggregator": overall_summary["aggregator"],
        "face_validation": fv_json,
        "indicator_scores": overall_summary["per_run_indicator_scores"],
        "indicators_used": copy.deepcopy(current_indicators),
    })

    # 4) Stop if no changes recommended
    if not fv_json.get("recommend_change", False):
        print(f"\n=== Stopping: no changes recommended after cycle {cycle_id}. ===\n")
        break

    # 5) If changes are recommended: update rubric (weights/indicators), then LOOP to re-score ALL runs again
    #    (We intentionally do not do per-run score adjustments here, because the user requirement is: apply changes then rescore everything.)
    iu_messages = [
        {"role": "system", "content": "You update transparent, reusable indicators with minimal drift."},
        {"role": "user", "content": "SIMULATION DESCRIPTION:\n" + simulation_description},
        {"role": "user", "content": "CURRENT INDICATORS:\n" + json.dumps(current_indicators, indent=2)},
        {"role": "user", "content": "FACE VALIDATION OUTPUT:\n" + json.dumps(fv_json, indent=2)},
        {"role": "user", "content": prompt_indicator_update()},
    ]

    iu_resp = client.chat.completions.create(model="gpt-5.1", messages=iu_messages)
    iu_json = safe_json_loads(iu_resp.choices[0].message.content)

    updated_indicators = iu_json["indicators"]
    updated_indicators = normalize_indicator_weights(updated_indicators)

    # switch rubric for next cycle
    current_indicators = [
        {"name": i["name"], "description": i["description"], "weight": float(i["weight"])}
        for i in updated_indicators
    ]

    print(f"Rubric updated. Re-scoring all runs in next cycle...\n")

print("\n=== Finished iterative full-run evaluation ===\n")
print(f"Cycles executed: {len(all_cycle_results)}")



=== Cycle 1: scoring all runs 1–21 ===

Run 1/21: indicator scoring done.
Run 2/21: indicator scoring done.
Run 3/21: indicator scoring done.
Run 4/21: indicator scoring done.
Run 5/21: indicator scoring done.
Run 6/21: indicator scoring done.
Run 7/21: indicator scoring done.
Run 8/21: indicator scoring done.
Run 9/21: indicator scoring done.
Run 10/21: indicator scoring done.
Run 11/21: indicator scoring done.
Run 12/21: indicator scoring done.
Run 13/21: indicator scoring done.
Run 14/21: indicator scoring done.
Run 15/21: indicator scoring done.
Run 16/21: indicator scoring done.
Run 17/21: indicator scoring done.
Run 18/21: indicator scoring done.
Run 19/21: indicator scoring done.
Run 20/21: indicator scoring done.
Run 21/21: indicator scoring done.

Cycle 1 overall credibility: 0.847

Face validation (cycle 1) summary:
No change to the rubric or scores appears warranted based on this batch. The main weakness (inconclusive leverage effect) is consistently and conservatively refl

## Prettify Batch logs

In [26]:
from pprint import pformat
from typing import Any

def pretty_py(obj: Any, *, width: int = 100, depth: int | None = None, compact: bool = False) -> str:
    return pformat(obj, width=width, depth=depth, compact=compact, sort_dicts=False)

# usage:
print(pretty_py(cycle_feedback_log, width=110, depth=None, compact=False))


[{'cycle_id': 1,
  'run_indices': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20],
  'overall_credibility': 0.8465238095238095,
  'aggregator': {'type': 'weighted_mean_of_indicator_means',
                 'note': 'Batch credibility = weighted arithmetic mean of per-indicator mean scores across '
                         'runs in this batch (including batch-mean run).'},
  'face_validation': {'short_summary': 'No change to the rubric or scores appears warranted based on this '
                                       'batch. The main weakness (inconclusive leverage effect) is '
                                       'consistently and conservatively reflected in lower scores for that '
                                       'indicator without distorting overall credibility.',
                      'issues_found': [],
                      'recommend_change': False,
                      'recommended_changes': []},
  'indicator_scores': [{'run_index': 0,
        